In [5]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer#needed to use iterative imputaion
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer#the 3 models of imputation used (simple contains mean and median)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score#cross validation scoring for classification sets
from sklearn.metrics import mean_squared_error#RootMeanSquaredError used to asses accuracy of Gene Expression
from sklearn.preprocessing import StandardScaler#Standard scalar used to improve accuracy of miltiple of my classification tasks
from sklearn.model_selection import GridSearchCV#final import used for gridsearch hyperparameter optimization to further improve accuracy on Sets 4,5

In [ ]:
#I went back and made the variables pretty in cell 1 as this was the code i featured in my presentation. I know i should do it like this always but my first draft was not nearly as readable and I resued many lines in each cell 
train_data_1 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData1.txt', delimiter='\t', header=None)#created variables to read the datasets that handles tabed spacing with the delimiter, specifies there is no header
train_label_1 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainLabel1.txt', header=None)
test_data_1 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData1.txt', delimiter='\t', header=None)

train_data_1.replace(1.0e+99, np.nan, inplace=True )#used .replace to get rid of the missing vaule (1.0e+99) with 'NaN' in the same place in both training and testing data 
test_data_1.replace(1.0e+99, np.nan, inplace=True)

simple_imputer = SimpleImputer(strategy='mean')#defined a variable to call a simple mean imputer
train_data_1_imputed = simple_imputer.fit_transform(train_data_1)#created new variables for the now imputed datasets for both training and testing data
test_data_1_imputed = simple_imputer.transform(test_data_1)

svm_model = SVC(kernel='linear',random_state=42)# initialized SVM with a linear kernal and the classic randomstate of 42. Ive used 42 ever since geeks for geeks taught me reproducabiluty but have never read the hitchickers guide to the galexy that it refrences

cv_splits = min(train_label_1[0].value_counts().min(), 5) #created a variable for how many splits to use in crossvalidation, selecting the minimum between the smallest class size in training labels 1
cross_val_scores = cross_val_score(svm_model, train_data_1_imputed, train_label_1.values.ravel(), cv=cv_splits)#variable to use the svm model on the imputed trainging data and label 1 for accuracy assesment

print(f"Cross-validation scores for Dataset 1 (using {cv_splits} splits): {cross_val_scores}")#print statment to make the output readable showing numebr of splits and each score
print(f"Mean accuracy: { cross_val_scores.mean():.2f}")# takes an avg of the folds scores and prints an average

svm_model.fit(train_data_1_imputed, train_label_1.values.ravel())#.fit calls the svm to train on the training data and label
predictions_1 =svm_model.predict(test_data_1_imputed)#created a variable where .predict calls for the trained model to make prediciton labels based on the test data 1

output_path = '/Users/noahrogers/desktop/ML_project/Outputs/RogersClassificaion1.txt'#created a variable to simplify saving the outputs
np.savetxt(output_path, predictions_1, fmt='%d')#call to save the predicion variable to the output path in integer format

Cross-validation scores for Dataset 1 (using 3 splits): [0.98 0.9  0.98]
Mean accuracy: 0.95


In [7]:
traind2 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData2.txt', delim_whitespace=True, header=None)#created variables to read the datasets with a whitespace delimieter because of dataformat, specifies there is no header
trainl2 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainLabel2.txt', header=None)
testd1 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData2.txt',  delim_whitespace=True, header=None)

traind2 = traind2.apply(pd.to_numeric, errors='coerce')#again because this dataset was in a different format the NaN replacement i used earlier wouldnt work and i found this pd.to.numeric coersion online that finally worked
testd1 = testd1.apply(pd.to_numeric, errors='coerce')#KNN imputers and RF classifiers need purly numerical data

knnimp = KNNImputer(n_neighbors=5)# variable for Knn imputer based on the 5 nearest neighbor
traind2imp = knnimp.fit_transform(traind2)#.fit_tranform perfomrs the imputation on the traing data
testd2 = knnimp.transform(testd1)#.fit applies the known imputation on test data

rf = RandomForestClassifier(random_state=42)#initilized the random forrest with the same random state for reproducabilty
cvs = cross_val_score( rf, traind2imp, trainl2.values.ravel(), cv=5)#calls for cross validation score for RF on training data and label 2

print(f"Cross-validation scores for Dataset 2 (using 5 splits): {cvs}")#Print statment for the 5 fold cvs
print(f"Mean accuracy: {cvs.mean():.2f}")#prints average score

rf.fit(traind2imp, trainl2.values.ravel())#rtraining the rf for prediciton
pred2 = rf.predict(testd2)#predicion call

out = '/Users/noahrogers/desktop/ML_project/Outputs/RogersClassification2.txt'
np.savetxt(out, pred2, fmt='%d' )

/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/1497299274.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  traind2 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData2.txt', delim_whitespace=True, header=None)#created variables to read the datasets with a whitespace delimieter because of dataformat, specifies there is no header
/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/1497299274.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  testd1 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData2.txt',  delim_whitespace=True, header=None)


Cross-validation scores for Dataset 2 (using 5 splits): [0.95 0.9  1.   0.8  0.9 ]
Mean accuracy: 0.91


In [8]:
traind3 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData3.txt', delim_whitespace=True, header=None)
trainl3 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainLabel3.txt', header=None)
testd3 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData3.txt', header=None)

imp3= SimpleImputer(strategy='mean')
traind3imp = imp3.fit_transform(traind3)
testd3imp = imp3.transform(testd3)

scale = StandardScaler()#created a variable to call the standard scalar
traind3sc = scale.fit_transform(traind3imp)#.fit_transform scalar to traind
testd3sc = scale.transform(testd3imp)#.transform applies scalar to testd

log = LogisticRegression(max_iter=10000, random_state=42)#variable to call the logistic regresison with 10000 max iterations and the same random state
cvs = cross_val_score(log, traind3sc, trainl3.values.ravel(), cv=5)

print(f"Cross-validation scores for Dataset 3 (using 5 splits): { cvs}")
print(f"Mean accuracy: {cvs.mean():.2f}")

log.fit(traind3sc, trainl3.values.ravel())
pred3 = log.predict(testd3sc)

out ='/Users/noahrogers/desktop/ML_project/Outputs/RogersClassification3.txt'
np.savetxt(out, pred3, fmt='%d')

Cross-validation scores for Dataset 3 (using 5 splits): [0.28015873 0.26269841 0.28095238 0.28253968 0.28095238]
Mean accuracy: 0.28


/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/3739758017.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  traind3 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData3.txt', delim_whitespace=True, header=None)


In [9]:
traind4 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData4.txt', delim_whitespace=True, header=None)
trainl4 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainLabel4.txt', header=None)
testd4 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData4.txt', delim_whitespace=True, header=None)

imp= SimpleImputer(strategy='mean')
traindimp = imp.fit_transform(traind4)
testd4imp = imp.transform(testd4)

#This is the model where i first tried out hyperparameter optimizaion. My understanding of this is limited to the reaserch i did for this project so i used a similar format as the one I was shown.
# I should have tried to set up parameters better suited to my needs in this dataset. which i believe played a role in not achieving the desired accuracy
param_grid = { #created a dicitonary with grid parameters for grid search
    'n_estimators': [100, 200],#range of number of trees
    'max_depth': [None, 10, 20],#range of the depth of these trees, none is unrestricted
    'min_samples_split': [2, 5],#min samples needed to split an interanl node
    'min_samples_leaf': [1, 2]#min samples needed to be leaf nodes
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)#created a variable to perform gridsearch with CV on the RF with the parametrs defined above. n_jobs=-1 uses all avaiblabe cpu cores for paralell processing
grid_search.fit(traindimp, trainl4.values.ravel())#.fit call to train based on each avaialable parameters

parameters = grid_search.best_params_# variable for the best parameter in gridsearch to print
finalscore = grid_search.best_score_#variable for the score of the best parameter
print(f"Best Parameters: {parameters}")#print statment to clearify the parameters used in the end
print(f"Best Cross-Validation Score: {finalscore:.2f}")#print statment of that score

rf_best = RandomForestClassifier(**parameters, random_state=42)#new iniitalization based on the new parameters
rf_best.fit(traindimp, trainl4.values.ravel())#.fit call for the new version
pred4 = rf_best.predict(testd4imp)

out ='/Users/noahrogers/desktop/ML_project/Outputs/RogersClassification4.txt'
np.savetxt(out, pred4, fmt='%d')

/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/3423826471.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  traind4 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData4.txt', delim_whitespace=True, header=None)
/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/3423826471.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  testd4 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData4.txt', delim_whitespace=True, header=None)


Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best Cross-Validation Score: 0.73


In [10]:
traind5 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData5.txt', delim_whitespace=True, header=None)
trainl5 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainLabel5.txt',  header=None)
testd5 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData5.txt', delim_whitespace=True, header=None)

traind5.replace(1.0e+99, np.nan, inplace=True)#original NaN missing handler
testd5.replace(1.0e+99, np.nan, inplace=True)

knnimp = KNNImputer(n_neighbors=5)
traind5imp = knnimp.fit_transform(traind5)
testd5imp = knnimp.transform(testd5)

scale = StandardScaler()
traind5sc = scale.fit_transform(traind5imp)
testd5sc = scale.transform(testd5imp)

param_grid = {#params for gridsearch and KNN
    'n_neighbors': [3, 5, 7, 9, 11],#num of neighbors
    'weights': ['uniform', 'distance'],#weigh neighbors evenly or by distance
    'metric': ['euclidean', 'manhattan', 'minkowski']#which distance formula is best
}

knn = KNeighborsClassifier()#initilize the KNN

cv_splits = min(trainl5[0].value_counts().min(), 5) #similar to dataset 1 assess the number of folds in CV min (between label and 5)
grid_search = GridSearchCV(knn, param_grid, cv=cv_splits, scoring='accuracy', n_jobs=-1)#grid search on knn, otherwise same format
grid_search.fit(traind5sc, trainl5.values.ravel())

parameters = grid_search.best_params_
finalscore = grid_search.best_score_
print(f"Best Parameters: {parameters}")
print(f"Best Cross-Validation Score: {finalscore:.2f}")

best_knn_model = KNeighborsClassifier(**parameters)
best_knn_model.fit(traind5sc, trainl5.values.ravel())

pred5 = best_knn_model.predict(testd5sc)

out ='/Users/noahrogers/desktop/ML_project/Outputs/RogersClassification5.txt'
np.savetxt(out, pred5, fmt='%d')

Best Parameters: {'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'distance'}
Best Cross-Validation Score: 0.54


/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/735538343.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  traind5 =pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TrainData5.txt', delim_whitespace=True, header=None)
/var/folders/94/_yvw94zx4sz4vqjh1bp9nrdw0000gn/T/ipykernel_57742/735538343.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  testd5 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/TestData5.txt', delim_whitespace=True, header=None)


In [11]:
data1 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/MissingData1.txt', sep='\t', header=None)#read gene expression file
data1.replace(1.00000000000000e+99, np.nan, inplace=True)#replace missing values 

knnimp = KNNImputer(n_neighbors=5)#KNN imputer for smaller set
data1imp = pd.DataFrame(knnimp.fit_transform(data1), columns=data1.columns)#imputed using a DF to maintain structire of output file

out ='/Users/noahrogers/desktop/ML_project/Outputs/RogersCompletedData1.txt'
data1imp.to_csv(out, sep='\t', index=False, header=False)#saved with.to_csv to maintain structure

In [12]:
data2 = pd.read_csv('/Users/noahrogers/desktop/ML_project/48506850Project/MissingData2.txt', sep='\t', header=None)
data2.replace(1.00000000000000e+99, np.nan, inplace=True)


iterativeimp = IterativeImputer(max_iter=10, random_state=0)
data2imp = pd.DataFrame(iterativeimp.fit_transform(data2), columns=data2.columns)

out ='/Users/noahrogers/desktop/ML_project/Outputs/RogersCompletedData2.txt'
data2imp.to_csv(out, sep='\t', index=False, header=False)